# Module 3: Discrete Term Structure (Bootstrapping)

Extracting zero-coupon spot rates from a par yield curve using recursive
bootstrapping. We verify against textbook examples and compare annual vs.
semi-annual handling.

In [ ]:
# ruff: noqa: E402
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from src.bootstrap.bootstrap import bootstrap_spot_rates
from src.instruments.bond import Bond
from src.utils.plotting import figure, finish_plot, set_theme

set_theme()
print("All imports OK")

## 1. Textbook Example: Upward-Sloping Par Yield Curve

This example mirrors the classic Fabozzi / Tuckman bootstrapping exercise.
A 5-year par yield curve is bootstrapped to reveal the spot rate curve.
Spot rates exceed par yields in an upward-sloping environment (coupon
effect).

In [ ]:
maturities = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
par_yields = np.array([0.030, 0.035, 0.040, 0.045, 0.050])

spot_annual = bootstrap_spot_rates(maturities, par_yields, freq=1)

pd.DataFrame(
    {
        "Maturity (yr)": maturities,
        "Par Yield (%)": par_yields * 100,
        "Spot Rate (%)": spot_annual * 100,
    }
).round(4)

The spot rates are higher than par yields for longer maturities — this is
the **coupon effect**: the par bond's coupons are discounted at lower
short-term rates, so the final cash flow must compensate with a higher
spot rate.

## 2. Verification: Re-Pricing Par Bonds at Spot Rates

Each bootstrapped spot rate should re-price its corresponding par bond to
exactly 100 (face value).

In [ ]:
def price_with_spot_curve(bond, spot_curve):
    """Price a bond using the spot curve for discounting."""
    periods = bond.periods
    freq = bond.freq
    pv = 0.0
    for k in range(1, periods + 1):
        t = k / freq
        z = np.interp(t, spot_curve[:, 0], spot_curve[:, 1])
        df = 1.0 / (1.0 + z) ** t
        if k == periods:
            pv += (bond.coupon_payment + bond.face_value) * df
        else:
            pv += bond.coupon_payment * df
    return pv


spot_curve = np.column_stack([maturities, spot_annual])
prices = []
for t, c in zip(maturities, par_yields, strict=True):
    b = Bond(face_value=100, coupon_rate=c, maturity=t, freq=1)
    p = price_with_spot_curve(b, spot_curve)
    prices.append(p)

pd.DataFrame(
    {
        "Maturity (yr)": maturities,
        "Re-Priced PV": np.round(prices, 8),
        "Par Value": 100.0,
        "Match": [abs(p - 100.0) < 1e-10 for p in prices],
    }
)

## 3. Semi-Annual Bootstrapping

Real Treasury bonds pay semi-annual coupons, requiring interpolation of
discount factors for cash flows between known maturity points.

In [ ]:
spot_semiannual = bootstrap_spot_rates(maturities, par_yields, freq=2)

pd.DataFrame(
    {
        "Maturity (yr)": maturities,
        "Par Yield (%)": par_yields * 100,
        "Spot Annual (%)": spot_annual * 100,
        "Spot Semi-Annual (%)": spot_semiannual * 100,
    }
).round(4)

The semi-annual spot rates differ slightly from annual due to the
interpolation of intermediate cash flows.

## 4. Flat Curve Validation

A flat par yield curve should produce flat spot rates (annual case).

In [ ]:
flat_par = np.full_like(maturities, 0.05)
flat_spot = bootstrap_spot_rates(maturities, flat_par, freq=1)

pd.DataFrame(
    {
        "Maturity (yr)": maturities,
        "Par Yield (%)": flat_par * 100,
        "Spot Rate (%)": flat_spot * 100,
    }
).round(6)

All spot rates exactly match the flat 5% par yield — confirming the
recursion is correct.

## 5. Visualisation: Par Yield vs. Spot Curve

In [ ]:
fig, ax = figure()

ax.plot(maturities, par_yields * 100, "o--", label="Par Yield Curve", linewidth=1.5)
ax.plot(maturities, spot_annual * 100, "s-", label="Spot Curve (Annual)", linewidth=2)
ax.plot(
    maturities,
    spot_semiannual * 100,
    "^-",
    label="Spot Curve (Semi-Annual)",
    linewidth=2,
)

ax.set_xlabel("Maturity (years)")
ax.set_ylabel("Yield (%)")
ax.set_title("Bootstrapped Spot Rates vs. Par Yields")
ax.legend()
ax.grid(True, alpha=0.3)

finish_plot(fig, "03-bootstrapping-spot-curve")